In [2]:
"""
Split dataset into train/test folders with stratified sampling.

Input structure:
    dataset/
        eyes/
            open/
                *.png
            closed/
                *.png
        mouth/
            open/
                *.png
            closed/
                *.png

Output structure:
    dataset_split/
        train/
            eyes/
                open/
                closed/
            mouth/
                open/
                closed/
        test/
            eyes/
                open/
                closed/
            mouth/
                open/
                closed/
"""

from __future__ import annotations

import random
import shutil
from pathlib import Path


# =========================
# Config
# =========================
INPUT_ROOT  = Path("../dataset")
OUTPUT_ROOT = Path("../dataset_split")

TEST_SPLIT  = 0.15   # 15% test, 85% train
SEED        = 42

ROIS = {
    "eyes":  ["open", "closed"],
    "mouth": ["open", "closed"],
}


# =========================
# Split
# =========================
def split_class(
    class_dir: Path,
    train_out: Path,
    test_out: Path,
    test_split: float,
    seed: int,
) -> dict[str, int]:
    """Copy files from one class folder into train/test output dirs."""
    images = sorted(class_dir.glob("*.png"))

    if not images:
        print(f"  WARNING: No .png files found in {class_dir}")
        return {"total": 0, "train": 0, "test": 0}

    random.seed(seed)
    random.shuffle(images)

    n_test  = max(1, int(len(images) * test_split))
    n_train = len(images) - n_test

    test_files  = images[:n_test]
    train_files = images[n_test:]

    train_out.mkdir(parents=True, exist_ok=True)
    test_out.mkdir(parents=True, exist_ok=True)

    for f in train_files:
        shutil.copy2(f, train_out / f.name)
    for f in test_files:
        shutil.copy2(f, test_out / f.name)

    return {"total": len(images), "train": n_train, "test": n_test}


def split_dataset(
    input_root: Path,
    output_root: Path,
    rois: dict[str, list[str]],
    test_split: float,
    seed: int,
) -> None:
    print(f"Input  : {input_root.absolute()}")
    print(f"Output : {output_root.absolute()}")
    print(f"Split  : {int((1 - test_split) * 100)}% train / {int(test_split * 100)}% test")
    print(f"Seed   : {seed}\n")

    for roi, classes in rois.items():
        print(f"--- {roi.upper()} ---")

        for class_name in classes:
            class_dir = input_root / roi / class_name

            if not class_dir.exists():
                print(f"  SKIP: {class_dir} does not exist")
                continue

            train_out = output_root / "train" / roi / class_name
            test_out  = output_root / "test"  / roi / class_name

            stats = split_class(class_dir, train_out, test_out, test_split, seed)

            print(
                f"  {class_name:<10} "
                f"total={stats['total']:>6} | "
                f"train={stats['train']:>6} | "
                f"test={stats['test']:>5}"
            )

        print()

    print("Done.")


if __name__ == "__main__":
    split_dataset(
        input_root=INPUT_ROOT,
        output_root=OUTPUT_ROOT,
        rois=ROIS,
        test_split=TEST_SPLIT,
        seed=SEED,
    )

Input  : c:\Users\User\Desktop\University\2026-Autumn\42028-DL-CNN\Assignments\FS\FatigueSense\notebooks\..\dataset
Output : c:\Users\User\Desktop\University\2026-Autumn\42028-DL-CNN\Assignments\FS\FatigueSense\notebooks\..\dataset_split
Split  : 85% train / 15% test
Seed   : 42

--- EYES ---
  open       total= 24755 | train= 21042 | test= 3713
  closed     total=  2701 | train=  2296 | test=  405

--- MOUTH ---
  open       total=   250 | train=   213 | test=   37
  closed     total= 13655 | train= 11607 | test= 2048

Done.
